In [0]:
import logging
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, DoubleType
from delta.tables import DeltaTable

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger(__name__)

BRONZE_PATH = "abfss://bronzelayer@cryptodl.dfs.core.windows.net/CSV_Streaming_Data_Source_3/datafiles"
SILVER_PATH = "abfss://silverlayer@cryptodl.dfs.core.windows.net/CSV_Streaming_Data_Source_3/"

spark = SparkSession.builder.getOrCreate()


log.info("Reading Bronze Source 3...")
df_bronze = spark.read.format("delta").load(BRONZE_PATH)
log.info(f"Bronze rows loaded: {df_bronze.count():,}")
df_bronze.printSchema()
display(df_bronze.limit(5))


In [0]:
# CELL 2 — Clean 
log.info("Cleaning and enriching data...")
df_clean = (
    df_bronze
    .dropDuplicates(["metric_id"])
    .withColumn("trade_date",          F.to_date(F.col("trade_date")))
    .withColumn("event_timestamp",     F.to_timestamp(F.col("event_timestamp")))
    .withColumn("ingestion_timestamp", F.to_timestamp(F.col("ingestion_timestamp")))
    .withColumn("active_addresses",    F.col("active_addresses").cast(LongType()))
    .withColumn("transaction_count",   F.col("transaction_count").cast(LongType()))
    .withColumn("network_fees",        F.col("network_fees").cast(DoubleType()))
    .withColumn("exchange_inflow",     F.col("exchange_inflow").cast(LongType()))
    .withColumn("exchange_outflow",    F.col("exchange_outflow").cast(LongType()))
    .withColumn("net_exchange_flow",   F.col("exchange_outflow") - F.col("exchange_inflow"))
    .withColumn("flow_signal",
        F.when(F.col("net_exchange_flow") > 0, "accumulation")
         .when(F.col("net_exchange_flow") < 0, "distribution")
         .otherwise("neutral")
    )
    .withColumn("silver_processed_at", F.current_timestamp())
    .drop("data_source")
)

log.info(f"After cleaning: {df_clean.count():,} rows")
display(df_clean.limit(5))


In [0]:
# CELL 3 — Data Quality Checks

log.info("Running data quality checks...")

log.info("Null check:")
df_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["metric_id","symbol","trade_date","active_addresses",
              "transaction_count","network_fees","exchange_inflow","exchange_outflow"]
]).show()

log.info("Flow signal distribution:")
df_clean.groupBy("flow_signal").count().orderBy("flow_signal").show()

log.info("Symbols check:")
df_clean.select("symbol").distinct().orderBy("symbol").show(30)


In [0]:
# CELL 4 — Write Silver Delta (Create or Merge)

df_final = df_clean.select(
    "metric_id", "symbol", "trade_date",
    "active_addresses", "transaction_count", "network_fees",
    "exchange_inflow", "exchange_outflow", "net_exchange_flow",
    "flow_signal", "event_timestamp", "ingestion_timestamp",
    "silver_processed_at"
)

log.info(f"Writing Silver Delta to: {SILVER_PATH}")

# Check if path exists first
try:
    dbutils.fs.ls(SILVER_PATH)
    path_exists = True
except Exception:
    path_exists = False

if path_exists and DeltaTable.isDeltaTable(spark, SILVER_PATH):
    log.info("Delta table exists — running MERGE...")
    (
        DeltaTable.forPath(spark, SILVER_PATH)
        .alias("target")
        .merge(df_final.alias("source"), "target.metric_id = source.metric_id")
        .whenMatchedUpdate(set={
            "active_addresses":    "source.active_addresses",
            "transaction_count":   "source.transaction_count",
            "network_fees":        "source.network_fees",
            "exchange_inflow":     "source.exchange_inflow",
            "exchange_outflow":    "source.exchange_outflow",
            "net_exchange_flow":   "source.net_exchange_flow",
            "flow_signal":         "source.flow_signal",
            "silver_processed_at": "source.silver_processed_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    log.info("MERGE complete")

else:
    log.info("Delta table not found — creating (first run)...")
    (
        df_final.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("symbol", "trade_date")
        .save(SILVER_PATH)
    )
    log.info("Delta table created")

In [0]:
log.info("Verifying Silver Delta...")
df_verify = spark.read.format("delta").load(SILVER_PATH)

log.info(f"Total rows    : {df_verify.count():,}")
log.info(f"Unique symbols: {df_verify.select('symbol').distinct().count()}")

display(
    df_verify.select(
        "symbol", "trade_date", "active_addresses",
        "transaction_count", "network_fees",
        "net_exchange_flow", "flow_signal"
    ).orderBy("symbol").limit(20)
)